# Solution D2 · What `predict_proba` means in a KNN

**Type**: Interpret

## Objective

Interpret the `predict_proba` of a KNN: why it takes few values, what effect it has on the log-loss and on the ROC curve, whether it is calibrated, and how a single prediction is "explained" by looking at the neighbors that voted.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

X, y = make_classification(3000, n_features=6, n_informative=3, n_redundant=0, n_clusters_per_class=1, class_sep=0.8, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)
scaler = StandardScaler().fit(X_tr)
Z_tr, Z_te = scaler.transform(X_tr), scaler.transform(X_te)

models = {
    "knn1": KNeighborsClassifier(1), "knn5": KNeighborsClassifier(5), "knn25": KNeighborsClassifier(25),
    "knn25_d": KNeighborsClassifier(25, weights="distance"), "lr": LogisticRegression(max_iter=1000),
}
probabilities = {name: m.fit(Z_tr, y_tr).predict_proba(Z_te)[:, 1] for name, m in models.items()}
knn25 = models["knn25"]
error_idx = int(np.flatnonzero(knn25.predict(Z_te) != y_te)[0])           # first test example that knn25 gets wrong


def ece(y, p, k=10):
    g = pd.DataFrame({"y": y, "p": p, "g": pd.qcut(p, k, duplicates="drop")}).groupby("g", observed=True).agg(n=("y", "size"), y=("y", "mean"), p=("p", "mean"))
    return float((g["n"] / g["n"].sum() * (g["p"] - g["y"]).abs()).sum())

## Commented solution

In [ ]:
n_values = {name: int(np.unique(p).size) for name, p in probabilities.items()}
print(n_values)

In [ ]:
table = pd.DataFrame(index=list(models), columns=["auc", "logloss", "roc_points", "ece"], dtype=float)
for name, p in probabilities.items():
    table.loc[name] = [roc_auc_score(y_te, p), log_loss(y_te, p), len(roc_curve(y_te, p)[0]), ece(y_te, p)]
table.round(3)

In [ ]:
why_logloss_is_high = "b"

With `k = 1` every probability is 0 or 1: when the neighbor is wrong (about one case in five) the probability assigned to the true class is zero and the log-loss clips it to `1e-15`, that is a loss of 34.5 per example. The mean
is dominated by those cases. It is a property of the metric applied to uncalibrated probabilities, not a defect of the ordering (the AUC of `knn1` is poor for another reason: it has only two values, that is a single threshold).

In [ ]:
distances, indices = knn25.kneighbors(Z_te[[error_idx]])
neighbors_table = pd.DataFrame({"index": indices[0], "distance": distances[0], "class": y_tr[indices[0]]})
votes = {int(c): int((neighbors_table["class"] == c).sum()) for c in (0, 1)}
predicted = max(votes, key=votes.get)
margin = votes[predicted] - votes[1 - predicted]
print(f"test example {error_idx}: true class {y_te[error_idx]}, predicted {predicted} | votes {votes} | margin {margin}")
neighbors_table.round(3).head(5)

## Check

In [ ]:
with verify():
    check(n_values["knn1"] == 2 and n_values["knn5"] == 6 and n_values["knn25"] == 26 and n_values["knn25_d"] > 100 and n_values["lr"] > 1000, "k+1 values for the uniform KNNs (2, 6, 26); over a hundred with 1/distance weights; thousands with the logistic", "np.unique(p).size")
    check(table.notna().all().all() and close(table.loc["knn5", "roc_points"], len(roc_curve(y_te, probabilities["knn5"])[0])), "complete table", "five models, four columns")
    check(table.loc["knn1", "logloss"] > 5 and table.loc["knn1", "logloss"] > 5 * table.loc["knn5", "logloss"] and table.loc["knn25", "logloss"] < table.loc["knn5", "logloss"] and table.loc["lr", "logloss"] < table.loc["knn25", "logloss"], "log-loss: over 5 with k=1 (more than 5 times that of k=5), then it drops with k=25 and the logistic is the best", "use sklearn.metrics.log_loss on the raw probabilities")
    check(table.loc["knn1", "roc_points"] == 3 and 10 < table.loc["knn25", "roc_points"] <= 27 and table.loc["lr", "roc_points"] > 50, "the ROC curve has 3 points for k=1, at most 27 for k=25 (scikit-learn drops collinear points), over 50 for the logistic", "len(roc_curve(y, p)[0])")
    check(table.loc["knn25", "auc"] > table.loc["knn1", "auc"] + 0.1 and table.loc["lr", "ece"] < table.loc["knn25", "ece"] / 1.5, "k = 25 has an AUC better than k = 1 by over 0.1; the logistic has an ECE at least 1.5 times better than k = 25", "compare auc and ece")
    check(table.loc["knn1", "ece"] < 0.05, "with k = 1 the ECE is small (below 0.05) but it is an illusion: the groups with equal probabilities collapse into one", "pd.qcut(..., duplicates='drop') with only two distinct values")
    check(why_logloss_is_high == "b", "correct answer: wrong 0/1 probabilities cost 34.5", "b")
    check(list(neighbors_table.columns) == ["index", "distance", "class"] and len(neighbors_table) == 25 and neighbors_table["distance"].is_monotonic_increasing and sum(votes.values()) == 25, "25 neighbors, increasing distances, 25 votes", "knn25.kneighbors(Z_te[[error_idx]])")
    check(margin == max(votes.values()) - min(votes.values()) and int(y_te[error_idx]) != max(votes, key=votes.get), "the margin is the difference in votes; the true class lost the vote", "votes of the predicted class minus votes of the other")
done()

## What you learned

- **The `predict_proba` of a uniform KNN takes only `k + 1` values** (0, 1/k, ..., 1): with `k = 5` that is 6. The ROC curve therefore has at most `k + 2` points and the AUC is a coarse estimate; with 1/distance weights the values become thousands (almost one per example), but the probabilities remain
  fractions of "weighted" neighbors, not risk estimates.
- **The log-loss of a KNN with small `k` is enormous**: with `k = 1` it is 5.3 (the logistic: 0.18), with `k = 5` 0.62, with `k = 25` 0.34. With `k = 1` the probabilities are exactly 0 or 1 and every error costs 34.5; with `k = 5` five agreeing neighbors are enough for a
  "sure" error. Don't compare the log-loss of a KNN with small `k` with that of other models without thinking about this effect.
- Calibration: the logistic has an ECE of 0.06, the KNN with `k = 25` of 0.12 (double: fractions of neighbors are not true probabilities) and the 1/distance weights don't change much (0.11). **Beware of `k = 1`, which looks the best (0.025):** with only two distinct values the ten groups
  collapse into one and the ECE compares only the overall mean. A calibration metric with quantile groups is unreliable with discrete scores. If you need true probabilities, recalibrate (`CalibratedClassifierCV`).
- KNN is **explainable by construction**: for every prediction you can show the `k` training examples that voted, with distances and classes. It is the only model of this course for which the explanation is "the most similar case to this one is that one".

## Common pitfall

Using the `predict_proba` of a KNN with small `k` as a "probability" in a risk or expected cost computation. They are fractions of a few neighbors: noisy, discrete and uncalibrated.

## Interview question

<details><summary>Can you give a probability with a KNN?</summary>

Yes, but with caution: the fraction of neighbors is a coarse, discrete estimate, poorly calibrated with small `k`. I use a larger `k`, check the calibration with a curve or an indicator like the ECE and, if needed, recalibrate it (sigmoid or isotonic)
on data not used for training. If the probabilities are central to the use case, I consider a model that estimates them directly, like the logistic.

</details>